In [ ]:
import os
import numpy as np
import laspy
from glob import glob
from tqdm import tqdm
from scipy.spatial import KDTree

In [ ]:
ORIGINAL_SCENE_DIR = ""
STITCHED_RESULTS_DIR = ""
OUTPUT_DIR = ""

In [ ]:
def propagate_and_filter():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    stitched_files = glob(os.path.join(STITCHED_RESULTS_DIR, "*_stitched.laz"))
    
    for stitched_path in stitched_files:
        base_name = os.path.basename(stitched_path).replace("_stitched.laz", "")
        original_match = glob(os.path.join(ORIGINAL_SCENE_DIR, f"{base_name}.la*"))
        
        if not original_match:
            continue
            
        original_path = original_match[0]
        print(f"Processing: {base_name}")
        
        src_las = laspy.read(stitched_path)
        src_pts = np.vstack((src_las.x, src_las.y, src_las.z)).T
        src_labels = np.array(src_las.ai_prediction)
        
        dst_las = laspy.read(original_path)
        
        mask = dst_las.classification != 3
        filtered_points = dst_las.points[mask]
        dst_pts = np.vstack((filtered_points.x, filtered_points.y, filtered_points.z)).T
        
        tree = KDTree(src_pts)
        _, indices = tree.query(dst_pts, k=1)
        final_labels = src_labels[indices]
        
        out_las = laspy.LasData(dst_las.header)
        out_las.points = filtered_points.copy()
        
        out_las.add_extra_dim(laspy.ExtraBytesParams(name="ai_prediction", type=np.int32))
        out_las.ai_prediction = final_labels
        
        output_path = os.path.join(OUTPUT_DIR, f"{base_name}_full_filtered.laz")
        out_las.write(output_path)

if __name__ == "__main__":
    propagate_and_filter()